# 1. 셀레니움
셀레니움(Selenium)은 파이썬 코드로 실제 웹 브라우저(Chrome, Edge 등)를 자동으로 제어할 수 있게 해주는 웹 자동화 라이브러리입니다. 일반적인 requests 크롤링은 서버에서 받은 HTML만 분석하지만, Selenium은 브라우저를 직접 실행하여 버튼 클릭, 검색 입력, 스크롤, 로그인, 페이지 이동 등의 사용자 동작을 그대로 수행할 수 있기 때문에 JavaScript로 동적으로 생성되는 데이터까지 가져올 수 있습니다. 따라서 멜론, 유튜브, 쇼핑몰처럼 JavaScript 렌더링이 많은 사이트의 크롤링이나 웹 자동화 테스트에서 매우 많이 사용됩니다. 보통 webdriver.Chrome()으로 브라우저를 실행하고, find_element()로 요소를 찾으며, click(), send_keys() 등을 통해 자동화 작업을 수행합니다.

### 1. JavaScript 때문에 requests로는 안 되는 페이지 예제
JavaScript가 실행되어야 화면에 데이터가 나타나는 구조

In [2]:
import requests
from bs4 import BeautifulSoup

url = "http://127.0.0.1:5500/7.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
fruits = soup.select(".fruit")
print(fruits)

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=5500): Max retries exceeded with url: /7.html (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=5500): Failed to establish a new connection: [WinError 10061] 대상 컴퓨터에서 연결을 거부했으므로 연결하지 못했습니다"))

### 2. 셀레니움으로 해결

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time



>👉 find_elements()는 Selenium에서 HTML 요소를 여러 개 찾을 때 사용하는 메서드입니다. 하나의 요소만 찾는 find_element()와 달리, 조건에 맞는 요소들을 리스트 형태로 모두 반환합니다. 예를 들어 여러 개의 이미지, 게시글, 테이블 행(tr) 등을 반복해서 크롤링할 때 매우 자주 사용됩니다. 찾는 방식은 By.ID, By.CLASS_NAME, By.CSS_SELECTOR, By.XPATH 등 다양한 선택자를 사용할 수 있으며, 반환 결과는 리스트이므로 for문으로 반복 처리하는 경우가 많습니다. 또한 find_element()는 요소를 찾지 못하면 에러가 발생하지만, find_elements()는 빈 리스트([])를 반환하기 때문에 반복 크롤링에서 더 안정적으로 사용되는 경우도 많습니다.

In [11]:
driver = webdriver.Chrome()
url = "http://127.0.0.1:5500/7.html"
driver.get(url)

# JavaScript 실행 대기
time.sleep(2)
fruits = driver.find_elements(By.CLASS_NAME, "fruit")

for fruit in fruits:
    print(fruit.text)

driver.quit()

사과
바나나
오렌지
딸기


# 2. 멜론 아티스트 곡 리스트 크롤링

### ※ xpath
XPath는 XML 또는 HTML 문서 내에서 특정 요소나 속성을 선택하기 위해 사용되는 경로 표현 언어입니다. 웹 크롤링이나 자동화 도구에서 주로 사용되며, 요소를 효율적으로 찾을 수 있도록 도와줍니다. 일반적인 XPath는 특정 위치나 속성을 기준으로 요소를 선택하는 상대적인 경로를 사용합니다. 또한 full xpath는 루트 요소에서 시작하여 대상 요소까지의 절대적인 경로를 나타냅니다. 따라서 문서 구조가 변경되면 경로가 깨질 가능성이 높습니다.

In [1]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [4]:
def melon_search_from_main(keyword):
    options = Options()
    # 반응형으로 바뀌면서 크롤링이 안될수도 있어서 최대화로 바꿈
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    # 최대 10초 까지 내가 원하는 값을 찾을때동안
    wait = WebDriverWait(driver, 10)

    data = []

    try:
        driver.get("https://www.melon.com/")
        time.sleep(2)
        # 멜론 페이지의 검색창 찾아오기
        search_box = wait.until(
            EC.presence_of_element_located((By.ID, "top_search"))
        )
        # 검색창 클리어 하기
        search_box.clear()
        # 받아온 키워드 입력
        search_box.send_keys(keyword)
        # 입력된 걸 엔터
        search_box.send_keys(Keys.ENTER)

        time.sleep(3)
        # 눌려질때 까지 기다리고 누른건 아니다
        song_tab = wait.until(
            EC.element_to_be_clickable(
                # 선택할 부분의 마우스 오른쪽 눌러서 copy - xpath를 복사하여 찾아감
                (By.XPATH, '//*[@id="divCollection"]/ul/li[3]/a/span')
            )
        )

        song_tab.click()
        time.sleep(3)

        song_table = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, '//*[@id="frm_defaultList"]/div/table')
            )
        )

        rows = song_table.find_elements(By.CSS_SELECTOR, "tbody tr")

        print("찾은 행 개수:", len(rows))

        for row in rows:
            try:
                cols = row.find_elements(By.TAG_NAME, "td")

                if len(cols) < 5:
                    continue

                # 곡명 td
                title_text = cols[2].text.strip()
                title_lines = [
                    line.strip()
                    for line in title_text.split("\n")
                    if line.strip()
                ]

                title = ""

                for line in title_lines:
                    if (
                        "재생" not in line
                        and "담기" not in line
                        and "상세정보" not in line
                        and not line.startswith("Title")
                    ):
                        title = line
                        break

        #         # Title 줄이 더 정확한 경우 보정
                for line in title_lines:
                    if line.startswith("Title "):
                        title = line.replace("Title ", "").strip()
                        break

        #         # 아티스트
                artist_text = cols[3].text.strip()
                artist_lines = [
                    line.strip()
                    for line in artist_text.split("\n")
                    if line.strip()
                ]
                artist = artist_lines[0] if artist_lines else ""

        #         # 앨범
                album_text = cols[4].text.strip()
                album_lines = [
                    line.strip()
                    for line in album_text.split("\n")
                    if line.strip()
                ]
                album = album_lines[0] if album_lines else ""

        #         # 좋아요 수
                like = ""
                try:
                    like = row.find_element(
                        By.CSS_SELECTOR,
                        "button.like span.cnt"
                    ).text.strip()
                except:
                    pass

                if title:
                    data.append({
                        "곡명": title,
                        "아티스트": artist,
                        "앨범": album,
                        "좋아요수": like
                    })

            except Exception as e:
                print("행 처리 오류:", e)

        df = pd.DataFrame(data)

        if not df.empty:
            df.index = df.index + 1

        file_name = f"melon_{keyword}_songs.csv"
        df.to_csv(file_name, encoding="utf-8-sig")

        print(f"CSV 저장 완료: {file_name}")
        print(f"총 {len(df)}곡 수집 완료")

        return df

    finally:
        driver.quit()

In [5]:
melon_search_from_main("신의키스")

찾은 행 개수: 6
CSV 저장 완료: melon_신의키스_songs.csv
총 6곡 수집 완료


,곡명,아티스트,앨범,좋아요수
1,이젠 내가 더 사랑해 (PJ1),신의키스,The Bucket List,5
2,헤어질 준비,신의키스,The Bucket List,5
3,너 없는 생일,신의키스,The Bucket List,4
4,버킷 리스트,신의키스,The Bucket List,5
5,서로를 가장 잘 아는 남,신의키스,The Bucket List,6
6,이젠 내가 더 사랑해 (PJ2),신의키스,The Bucket List,4


In [9]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

def extract_current_page(driver, wait):
    data = []

    try:
        song_table = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, '//*[@id="frm_defaultList"]/div/table')
            )
        )
    except TimeoutException:
        return []

    rows = song_table.find_elements(By.CSS_SELECTOR, "tbody tr")

    for row in rows:
        cols = row.find_elements(By.TAG_NAME, "td")

        if len(cols) < 5:
            continue

        title_lines = [
            line.strip()
            for line in cols[2].text.split("\n")
            if line.strip()
        ]

        title = ""

        for line in title_lines:
            if line.startswith("Title "):
                title = line.replace("Title ", "").strip()
                break

        if not title:
            for line in title_lines:
                if (
                    "재생" not in line
                    and "담기" not in line
                    and "상세정보" not in line
                    and not line.startswith("Title")
                ):
                    title = line
                    break

        artist_lines = [
            line.strip()
            for line in cols[3].text.split("\n")
            if line.strip()
        ]
        artist = artist_lines[0] if artist_lines else ""

        album_lines = [
            line.strip()
            for line in cols[4].text.split("\n")
            if line.strip()
        ]
        album = album_lines[0] if album_lines else ""

        try:
            like = row.find_element(
                By.CSS_SELECTOR,
                "button.like span.cnt"
            ).text.strip()
        except:
            like = ""

        if title:
            data.append({
                "곡명": title,
                "아티스트": artist,
                "앨범": album,
                "좋아요수": like
            })

    return data





In [ ]:
def melon_search_all_pages(keyword, max_page=30):
    options = Options()
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    all_data = []

    try:
        driver.get("https://www.melon.com/")
        time.sleep(2)

        search_box = wait.until(
            EC.presence_of_element_located((By.ID, "top_search"))
        )

        search_box.clear()
        search_box.send_keys(keyword)
        search_box.send_keys(Keys.ENTER)

        time.sleep(3)

        song_tab = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, '//*[@id="divCollection"]/ul/li[3]/a/span')
            )
        )

        song_tab.click()
        time.sleep(3)

        for page in range(1, max_page + 1):
            page_data = extract_current_page(driver, wait)

            if not page_data:
                print(f"{page}페이지 데이터가 없어 종료합니다.")
                break

            all_data.extend(page_data)
            print(f"{page}페이지 크롤링 완료: {len(page_data)}곡")

            next_start_index = page * 50 + 1

            try:
                # 자바스크립트를 실행 해주는 함수
                driver.execute_script(
                    f"pageObj.sendPage('{next_start_index}');"
                )
                time.sleep(3)

            except Exception as e:
                print("다음 페이지 이동 실패:", e)
                break

        df = pd.DataFrame(all_data)

        if not df.empty:
            df = df.drop_duplicates(
                subset=["곡명", "아티스트", "앨범"]
            )
            df.index = df.index + 1

        file_name = f"melon_{keyword}_all_songs.csv"

        excel_file_name = "banapresso_store.xlsx"

        with pd.ExcelWriter(
            excel_file_name,
            engine="openpyxl",
            mode="a",
            if_sheet_exists="replace"
        ) as writer:
            df.to_excel(
                writer,
                sheet_name="멜론",
                index=False
            )

        print(f"CSV 저장 완료: {file_name}")
        print(f"총 {len(df)}곡 수집 완료")

        return df

    finally:
        driver.quit()

In [11]:
melon_search_all_pages("조용필")

1페이지 크롤링 완료: 50곡
2페이지 크롤링 완료: 50곡
3페이지 크롤링 완료: 50곡
4페이지 크롤링 완료: 50곡
5페이지 크롤링 완료: 50곡
6페이지 크롤링 완료: 50곡
7페이지 크롤링 완료: 50곡
8페이지 크롤링 완료: 50곡
9페이지 크롤링 완료: 50곡
10페이지 크롤링 완료: 50곡
11페이지 크롤링 완료: 50곡
12페이지 크롤링 완료: 50곡
13페이지 크롤링 완료: 50곡
14페이지 크롤링 완료: 50곡
15페이지 크롤링 완료: 50곡
16페이지 크롤링 완료: 50곡
17페이지 크롤링 완료: 50곡
18페이지 크롤링 완료: 50곡
19페이지 크롤링 완료: 50곡
20페이지 크롤링 완료: 50곡
21페이지 크롤링 완료: 50곡
22페이지 크롤링 완료: 50곡
23페이지 크롤링 완료: 50곡
24페이지 크롤링 완료: 50곡
25페이지 크롤링 완료: 50곡
26페이지 크롤링 완료: 20곡
27페이지 데이터가 없어 종료합니다.
CSV 저장 완료: melon_조용필_all_songs.csv
총 1270곡 수집 완료


,곡명,아티스트,앨범,좋아요수
1,바람의 노래,조용필,조용필 16집,"26,576"
2,꿈,조용필,The Dreams,"16,275"
3,HOT 잊혀진 사랑,조용필,30주년 기념 음반 Part 1,"2,414"
4,Bounce,조용필,Hello,"40,021"
5,HOT 단발머리,조용필,조용필 1집 (Remastered),"11,088"
...,...,...,...,...
1266,님이여,조용필,기쁜 우리 젊은날의 가요 4집,15
1267,마음속의 그림자,조용필,기쁜 우리 젊은날의 가요 4집,14
1268,세월이 가면,조용필,기쁜 우리 젊은날의 가요 1집,24
1269,Bounce - 조용필 (MR),음악상자,음악상자 12집,3


# 스타벅스 서울 전체 매장 크롤링

In [18]:
import re
from selenium.webdriver import ActionChains
from bs4 import BeautifulSoup



In [11]:
def fetch_starbucks():
    url = "https://www.starbucks.co.kr/index.do"

    driver = webdriver.Chrome()
    driver.maximize_window()

    driver.get(url)
    time.sleep(2)

    
    # 메뉴 이동
    action = ActionChains(driver)

    first_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#gnb > div > nav > div > ul > li.gnb_nav03"
    )

    second_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#gnb > div > nav > div > ul > li.gnb_nav03 > div > div > div > ul:nth-child(1) > li:nth-child(3) > a"
    )
    # 
    action.move_to_element(first_tag) \
          .move_to_element(second_tag) \
          .click() \
          .perform()

    # # 서울 선택
    seoul_tag = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((
            By.CSS_SELECTOR,
            "#container > div > form > fieldset > div > section > article.find_store_cont > article > article:nth-child(4) > div.loca_step1 > div.loca_step1_cont > ul > li:nth-child(1) > a"
        ))
    )

    seoul_tag.click()

    # # 구 목록 로딩 대기
    WebDriverWait(driver, 5).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "set_gugun_cd_btn")
        )
    )

    gu_elements = driver.find_elements(
        By.CLASS_NAME,
        "set_gugun_cd_btn"
    )

    # # 전체 선택
    gu_elements[0].click()

    # 매장 목록 로딩 대기
    WebDriverWait(driver, 5).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "quickResultLstCon")
        )
    )

    # # HTML 가져오기
    req = driver.page_source

    soup = BeautifulSoup(req, "html.parser")

    stores = soup.find(
        'ul',
        'quickSearchResultBoxSidoGugun'
    ).find_all('li')

    # 데이터 저장 리스트
    store_list = []
    addr_list = []
    lat_list = []
    lng_list = []

    # # 데이터 추출
    for store in stores:

        store_name = store.find("strong").text

        store_addr = store.find("p").text

        # 전화번호 제거
        store_addr = re.sub(
            r'\d{4}-\d{4}$',
            '',
            store_addr
        ).strip()

        store_lat = store['data-lat']
        store_lng = store['data-long']

        store_list.append(store_name)
        addr_list.append(store_addr)
        lat_list.append(store_lat)
        lng_list.append(store_lng)

    # 데이터프레임 생성
    df = pd.DataFrame({
        'store': store_list,
        'addr': addr_list,
        'lat': lat_list,
        'lng': lng_list
    })

    driver.quit()

    return df


# 함수 실행
starbucks_df = fetch_starbucks()

# CSV 저장
with pd.ExcelWriter(
    "banapresso_store.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    starbucks_df.to_excel(
        writer,
        sheet_name="스타벅스",
        index=False
    )


print("데이터가 starbucks_seoul.csv 파일로 저장되었습니다.")
print(starbucks_df.head())

데이터가 starbucks_seoul.csv 파일로 저장되었습니다.
       store                        addr         lat          lng
0  역삼아레나빌딩       서울특별시 강남구 언주로 425 (역삼동)   37.501087   127.043069
1   논현역사거리      서울특별시 강남구 강남대로 538 (논현동)   37.510178   127.022223
2  신사역성일빌딩      서울특별시 강남구 강남대로 584 (논현동)  37.5139309  127.0206057
3   국기원사거리      서울특별시 강남구 테헤란로 125 (역삼동)   37.499517   127.031495
4   대치재경빌딩    서울특별시 강남구 남부순환로 2947 (대치동)   37.494668   127.062583


In [6]:
import time
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup


def fetch_starbucks():
    url = "https://www.starbucks.co.kr/index.do"

    driver = webdriver.Chrome()
    driver.maximize_window()

    driver.get(url)
    time.sleep(2)

    # 메뉴 이동
    action = ActionChains(driver)

    first_tag = driver.find_element(
        By.CSS_SELECTOR,
        "body > div.bp-nav > div > div > div:nth-child(2) > a"
    )

    second_tag = driver.find_element(
        By.CSS_SELECTOR,
        "body > div.bp-nav > div > div > div:nth-child(2) > div > a:nth-child(1)"
    )

    action.move_to_element(first_tag) \
          .move_to_element(second_tag) \
          .click() \
          .perform()

    # 서울 선택
    seoul_tag = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((
            By.CSS_SELECTOR,
            "#container > div > form > fieldset > div > section > article.find_store_cont > article > article:nth-child(4) > div.loca_step1 > div.loca_step1_cont > ul > li:nth-child(1) > a"
        ))
    )

    seoul_tag.click()

    # 구 목록 로딩 대기
    WebDriverWait(driver, 5).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "set_gugun_cd_btn")
        )
    )

    gu_elements = driver.find_elements(
        By.CLASS_NAME,
        "set_gugun_cd_btn"
    )

    # 전체 선택
    gu_elements[0].click()

    # 매장 목록 로딩 대기
    WebDriverWait(driver, 5).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "quickResultLstCon")
        )
    )

    # HTML 가져오기
    req = driver.page_source

    soup = BeautifulSoup(req, "html.parser")

    stores = soup.find(
        'ul',
        'quickSearchResultBoxSidoGugun'
    ).find_all('li')

    # 데이터 저장 리스트
    store_list = []
    addr_list = []
    lat_list = []
    lng_list = []

    # 데이터 추출
    for store in stores:

        store_name = store.find("strong").text

        store_addr = store.find("p").text

        # 전화번호 제거
        store_addr = re.sub(
            r'\d{4}-\d{4}$',
            '',
            store_addr
        ).strip()

        store_lat = store['data-lat']
        store_lng = store['data-long']

        store_list.append(store_name)
        addr_list.append(store_addr)
        lat_list.append(store_lat)
        lng_list.append(store_lng)

    # 데이터프레임 생성
    df = pd.DataFrame({
        'store': store_list,
        'addr': addr_list,
        'lat': lat_list,
        'lng': lng_list
    })

    driver.quit()

    return df


# 함수 실행
starbucks_df = fetch_starbucks()

# CSV 저장
starbucks_df.to_csv(
    "starbucks_seoul.csv",
    index=False,
    encoding='utf-8-sig'
)

print("데이터가 starbucks_seoul.csv 파일로 저장되었습니다.")
print(starbucks_df.head())

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"body > div.bp-nav > div > div > div:nth-child(2) > a"}
  (Session info: chrome=149.0.7827.156); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff702e13fa5+14925]
	chromedriver!GetHandleVerifier [0x7ff702e14000+14980]
	chromedriver!(No symbol) [0x7ff70295793d]
	chromedriver!(No symbol) [0x7ff7029b1aad]
	chromedriver!(No symbol) [0x7ff7029b1dac]
	chromedriver!(No symbol) [0x7ff702a027d7]
	chromedriver!(No symbol) [0x7ff7029ff39b]
	chromedriver!(No symbol) [0x7ff7029a401c]
	chromedriver!(No symbol) [0x7ff7029a4f43]
	chromedriver!GetHandleVerifier [0x7ff7033f7591+5f7f11]
	chromedriver!GetHandleVerifier [0x7ff7033f1902+5f2282]
	chromedriver!GetHandleVerifier [0x7ff703417115+617a95]
	chromedriver!GetHandleVerifier [0x7ff702e31dce+3274e]
	ch

In [12]:
fetch_banapress()

In [ ]:
import time
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup

url = "https://banapresso.com/"

driver = webdriver.Chrome()
driver.maximize_window()

driver.get(url)
time.sleep(2)

# 메뉴 이동
action = ActionChains(driver)

first_tag = driver.find_element(
    By.CSS_SELECTOR,
    "#wrap > header > div > ul > li:nth-child(2) > a"
)

second_tag = driver.find_element(
    By.CSS_SELECTOR,
    "#wrap > header > div > ul > li:nth-child(2) > ul > li:nth-child(1) > a > span"
)

action.move_to_element(first_tag) \
      .move_to_element(second_tag) \
      .click() \
      .perform()

# 위치 허용 선택
use_tag = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((
        By.CSS_SELECTOR,
        "#root > div.sc-ff716c63-0.fOAigp > div > div.p_btm"
    ))
)

use_tag.click()

# 매장 목록 로딩 대기
WebDriverWait(driver, 10).until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, "li.sc-4ba45bc0-0")
    )
)

# HTML 가져오기
req = driver.page_source

soup = BeautifulSoup(req, "html.parser")

stores = soup.select("li.sc-4ba45bc0-0")

# 데이터 저장 리스트
store_list = []
addr_list = []
time_list = []
parking_list = []

# 데이터 추출
for store in stores:
    name_tag = store.select_one(".name")
    
    addr_tag = store.select_one(".address")
    time_tag = store.select_one(".store-time-wrap .time.open p:nth-of-type(2)")
    parking_tag = store.select_one(".store-time-wrap .parking span")
    print(parking_tag)

    store_name = name_tag.get_text(strip=True) if name_tag else ""
    store_addr = addr_tag.get_text(strip=True) if addr_tag else ""
    store_time = time_tag.get_text(strip=True) if time_tag else ""
    parking = time_tag.get_text(strip=True) if parking_tag else ""

    store_list.append(store_name)
    addr_list.append(store_addr)
    time_list.append(store_time)
    parking_list.append(parking)

# # 데이터프레임 생성
# df = pd.DataFrame({
#     'store': store_list,
#     'addr': addr_list,
#     'time': time_list,
#     'parking': parking,
# })

# driver.quit()

# print(df)
# print("총 매장 수:", len(df))

# df.to_csv(
#     "banapresso_store.csv",
#     index=False,
#     encoding="utf-8-sig"
# )


<p class="name">가락몰점</p>
<p class="name">가산디지털단지역점</p>
<p class="name">가산안양천점</p>
<p class="name">가산어반워크점</p>
<p class="name">가산우림라이온스점</p>
<p class="name">가산파트너스타워점</p>
<p class="name">가산하이엔드10차점</p>
<p class="name">가산IT캐슬점</p>
<p class="name">가산KM타워점</p>
<p class="name">강남구청점</p>


In [8]:
import time
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup

url = "https://banapresso.com/"

driver = webdriver.Chrome()
driver.maximize_window()

driver.get(url)
time.sleep(2)

# 메뉴 이동
action = ActionChains(driver)

first_tag = driver.find_element(
    By.CSS_SELECTOR,
    "#wrap > header > div > ul > li:nth-child(2) > a"
)

second_tag = driver.find_element(
    By.CSS_SELECTOR,
    "#wrap > header > div > ul > li:nth-child(2) > ul > li:nth-child(1) > a > span"
)

action.move_to_element(first_tag) \
      .move_to_element(second_tag) \
      .click() \
      .perform()

# 위치 허용 선택
use_tag = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((
        By.CSS_SELECTOR,
        "#root > div.sc-ff716c63-0.fOAigp > div > div.p_btm"
    ))
)

use_tag.click()

# 매장 목록 로딩 대기
WebDriverWait(driver, 10).until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, "li.sc-4ba45bc0-0")
    )
)

# 스크롤 끝까지 내리기
while True:
    stores = driver.find_elements(By.CSS_SELECTOR, "li.sc-4ba45bc0-0")
    current_count = len(stores)

    driver.execute_script(
        "arguments[0].scrollIntoView();",
        stores[-1]
    )

    time.sleep(1.5)

    stores = driver.find_elements(By.CSS_SELECTOR, "li.sc-4ba45bc0-0")
    new_count = len(stores)

    # 더 이상 매장 수가 늘어나지 않으면 종료
    if new_count == current_count:
        break

# HTML 가져오기
req = driver.page_source

soup = BeautifulSoup(req, "html.parser")

stores = soup.select("li.sc-4ba45bc0-0")

# 데이터 저장 리스트
store_list = []
addr_list = []
time_list = []
parking_list = []

# 데이터 추출
for store in stores:
    name_tag = store.select_one(".name")
    addr_tag = store.select_one(".address")
    time_tag = store.select_one(".store-time-wrap .time.open p:nth-of-type(2)")
    parking_tag = store.select_one(".store-time-wrap .parking span")

    store_name = name_tag.get_text(strip=True) if name_tag else ""
    store_addr = addr_tag.get_text(strip=True) if addr_tag else ""
    store_time = time_tag.get_text(strip=True) if time_tag else ""
    parking = parking_tag.get_text(strip=True) if parking_tag else ""

    store_list.append(store_name)
    addr_list.append(store_addr)
    time_list.append(store_time)
    parking_list.append(parking)

# 데이터프레임 생성
df = pd.DataFrame({
    '매장명': store_list,
    '주소': addr_list,
    '영업시간': time_list,
    '주차여부': parking_list,
})

driver.quit()

print(df)
print("총 매장 수:", len(df))

df.to_excel(
    "banapresso_store.xlsx",
    index=False
)


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=149.0.7827.156)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff702e13fa5+14925]
	chromedriver!GetHandleVerifier [0x7ff702e14000+14980]
	chromedriver!(No symbol) [0x7ff70295793d]
	chromedriver!(No symbol) [0x7ff70292e362]
	chromedriver!(No symbol) [0x7ff7029e1846]
	chromedriver!(No symbol) [0x7ff7029fec82]
	chromedriver!(No symbol) [0x7ff7029a401c]
	chromedriver!(No symbol) [0x7ff7029a4f43]
	chromedriver!GetHandleVerifier [0x7ff7033f7591+5f7f11]
	chromedriver!GetHandleVerifier [0x7ff7033f1902+5f2282]
	chromedriver!GetHandleVerifier [0x7ff703417115+617a95]
	chromedriver!GetHandleVerifier [0x7ff702e31dce+3274e]
	chromedriver!GetHandleVerifier [0x7ff702e3a82c+3b1ac]
	chromedriver!GetHandleVerifier [0x7ff702e1d744+1e0c4]
	chromedriver!GetHandleVerifier [0x7ff702e1d8d4+1e254]
	chromedriver!GetHandleVerifier [0x7ff702e01447+1dc7]
	KERNEL32!BaseThreadIn

In [9]:
with pd.ExcelWriter(
    "banapresso_store.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    starbucks_df.to_excel(
        writer,
        sheet_name="스타벅스",
        index=False
    )

print("스타벅스 데이터가 banapresso_store.xlsx 파일의 두 번째 탭에 저장되었습니다.")
print(starbucks_df.head())

PermissionError: [Errno 13] Permission denied: 'banapresso_store.xlsx'